In [ ]:
from fastapi import APIRouter, HTTPException, status, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials

from app.schemas.auth import (
    RegisterRequest,
    LoginRequest,
    UserResponse,
    TokenResponse,
)
from app.core.security import (
    hash_password,
    verify_password,
    create_access_token,
    decode_access_token,
)
from app.utils.helpers import generate_id, utc_now


router = APIRouter(
    prefix="/auth",
    tags=["Authentication"],
)


security = HTTPBearer()


def get_database():
    """
    Return the MongoDB database attached to the FastAPI application.

    The database is initialized by notebooks/run_backend.ipynb.
    """
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


@router.post(
    "/register",
    response_model=TokenResponse,
    status_code=status.HTTP_201_CREATED,
)
async def register(request: RegisterRequest):
    """
    Register a new learner.

    Flow:
    1. Validate RegisterRequest through Pydantic.
    2. Check whether the email already exists.
    3. Hash the password.
    4. Create the user in MongoDB.
    5. Generate a JWT.
    6. Return the token and user.
    """

    database = get_database()
    users = database.collection("users")

    # Normalize email so registration/login are case-insensitive.
    email = str(request.email).lower().strip()

    # Check for an existing account.
    existing_user = users.find_one({"email": email})

    if existing_user is not None:
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="An account with this email already exists.",
        )

    # Create the user document.
    user_id = generate_id()

    user_document = {
        "id": user_id,
        "email": email,
        "name": request.name.strip(),
        "password_hash": hash_password(request.password),
        "role": "student",
        "created_at": utc_now(),
        "updated_at": utc_now(),
    }

    try:
        users.insert_one(user_document)
    except Exception as exc:
        # MongoDB has a unique index on email.
        # This protects against a race condition where two registrations
        # use the same email simultaneously.
        if "duplicate key" in str(exc).lower():
            raise HTTPException(
                status_code=status.HTTP_409_CONFLICT,
                detail="An account with this email already exists.",
            )

        raise

    user = UserResponse(
        id=user_id,
        email=email,
        name=user_document["name"],
        role=user_document["role"],
    )

    access_token = create_access_token(
        {
            "sub": user_id,
            "email": email,
            "role": user_document["role"],
        }
    )

    return TokenResponse(
        access_token=access_token,
        token_type="bearer",
        user=user,
    )


@router.post(
    "/login",
    response_model=TokenResponse,
)
async def login(request: LoginRequest):
    """
    Authenticate a learner and return an access token.
    """

    database = get_database()
    users = database.collection("users")

    email = str(request.email).lower().strip()

    user_document = users.find_one({"email": email})

    if user_document is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid email or password.",
        )

    if not verify_password(
        request.password,
        user_document["password_hash"],
    ):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid email or password.",
        )

    user = UserResponse(
        id=user_document["id"],
        email=user_document["email"],
        name=user_document["name"],
        role=user_document.get("role", "student"),
    )

    access_token = create_access_token(
        {
            "sub": user.id,
            "email": user.email,
            "role": user.role,
        }
    )

    return TokenResponse(
        access_token=access_token,
        token_type="bearer",
        user=user,
    )


@router.get(
    "/me",
    response_model=UserResponse,
)
async def get_current_user(
    credentials: HTTPAuthorizationCredentials = Depends(security),
):
    """
    Return the currently authenticated user.
    """

    token = credentials.credentials

    try:
        payload = decode_access_token(token)
    except Exception:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or expired token.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    user_id = payload.get("sub")

    if not user_id:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid authentication token.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    database = get_database()
    users = database.collection("users")

    user_document = users.find_one({"id": user_id})

    if user_document is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="User no longer exists.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    return UserResponse(
        id=user_document["id"],
        email=user_document["email"],
        name=user_document["name"],
        role=user_document.get("role", "student"),
    )